# Day 2 — Tool calling: the model asks, your code runs
**Theory:** [day-02-tool-calling.md](../theory/day-02-tool-calling.md) · ⏱ ~60 min · Level: easy → medium

**You will:**
1. Do one tool call fully by hand and inspect every block
2. Handle **parallel** tool calls and **errors**
3. Use **strict** schemas and **structured outputs**
4. Run a tool-design A/B test (bad tool vs good tool)

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("ANTHROPIC_API_KEY", UserSecretsClient().get_secret("ANTHROPIC_API_KEY"))
    except Exception as e:
        print("Add ANTHROPIC_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. A real tool over the ShopEase database
The function is plain Python. The *definition* (name, description, schema) is all the model sees.

In [ ]:
import json, sqlite3
DB = "projects/shopease-support-agent/data/shopease.db"

def get_order(order_id: str) -> str:
    row = sqlite3.connect(DB).execute(
        "SELECT id, item, amount, status, delivered_on FROM orders WHERE id=?", (order_id,)).fetchone()
    if not row:
        return f"Error: no order {order_id}. Order ids look like 'O2001'."
    return json.dumps(dict(zip(["id", "item", "amount", "status", "delivered_on"], row)))

GET_ORDER = {
    "name": "get_order",
    "description": "Look up one ShopEase order by id (e.g. 'O2001'). Returns item, amount, status and delivery date.",
    "strict": True,
    "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                     "required": ["order_id"], "additionalProperties": False},
}
print(get_order("O2001"))

## 2. One round trip, by hand

In [ ]:
messages = [{"role": "user", "content": "What's the status of order O2001?"}]
r = client.messages.create(model=MODEL, max_tokens=2000, tools=[GET_ORDER], messages=messages,
                           output_config={"effort": "low"})
tracker.add(MODEL, r.usage, "tool call 1")
print("stop_reason:", r.stop_reason)
for b in r.content:
    print(" ", b.type, "→", getattr(b, "name", ""), getattr(b, "input", ""), (getattr(b, "text", "") or "")[:80])

In [ ]:
call = next(b for b in r.content if b.type == "tool_use")
result = get_order(**call.input)                      # YOUR code runs the tool
messages += [{"role": "assistant", "content": r.content},  # reply exactly as received
             {"role": "user", "content": [{"type": "tool_result", "tool_use_id": call.id, "content": result}]}]
final = client.messages.create(model=MODEL, max_tokens=2000, tools=[GET_ORDER], messages=messages,
                               output_config={"effort": "low"})
tracker.add(MODEL, final.usage, "tool call 2")
print(final.stop_reason, "→", text_of(final))

## 3. Parallel calls + errors as feedback
Ask about three orders, one of which doesn't exist. Run all calls, return **all results in ONE message**,
and mark the failure with `is_error: True` so the model can recover.

In [ ]:
def run_tools(response):
    results = []
    for b in response.content:
        if b.type == "tool_use":
            out = get_order(**b.input)
            print(f"   🔧 {b.name}({b.input}) → {out[:60]}")
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": out,
                            "is_error": out.startswith("Error")})
    return results

messages = [{"role": "user", "content": "Compare orders O2001, O2004 and O9999 in one short table."}]
for step in range(5):
    r = client.messages.create(model=MODEL, max_tokens=3000, tools=[GET_ORDER], messages=messages,
                               output_config={"effort": "low"})
    tracker.add(MODEL, r.usage, f"parallel step {step}")
    messages.append({"role": "assistant", "content": r.content})
    n_calls = sum(b.type == "tool_use" for b in r.content)
    print(f"step {step}: {n_calls} tool call(s) in one reply")
    if r.stop_reason != "tool_use":
        break
    messages.append({"role": "user", "content": run_tools(r)})
print("\n" + text_of(r))

## 4. Structured outputs: guaranteed-valid JSON
Perfect for classification and extraction. `messages.parse` validates against a Pydantic model.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Ticket(BaseModel):
    category: Literal["order_status", "refund", "warranty", "other"]
    order_id: str | None
    urgency: Literal["low", "medium", "high"]

tickets = ["My watch from order O2001 stopped working!! need help asap",
           "where's my package O2003, it's been weeks",
           "do you sell gift cards?"]
for t in tickets:
    r = client.messages.parse(model=CHEAP_MODEL, max_tokens=500, output_format=Ticket,
                              messages=[{"role": "user", "content": t}])
    tracker.add(CHEAP_MODEL, r.usage, "parse")
    print(f"{r.parsed_output!r:<75} ← {t}")

## 5. Tool design A/B: "dump everything" vs "task-shaped search"
Same question, two tool designs. Compare tokens and correctness.

In [ ]:
from course_utils import run_agent

def list_all_orders() -> str:
    rows = sqlite3.connect(DB).execute("SELECT * FROM orders").fetchall()
    return json.dumps(rows)

def search_orders(customer_id: str, status: str = "") -> str:
    q, args = "SELECT id, item, status FROM orders WHERE customer_id=?", [customer_id]
    if status:
        q, args = q + " AND status=?", args + [status]
    return json.dumps(sqlite3.connect(DB).execute(q, args).fetchall())

TOOL_A = [{"name": "list_all_orders", "description": "Return every order in the database.",
           "input_schema": {"type": "object", "properties": {}}}]
TOOL_B = [{"name": "search_orders", "description": "Search one customer's orders, optionally by status "
           "(processing/shipped/delivered/returned).",
           "input_schema": {"type": "object", "properties": {"customer_id": {"type": "string"}, "status": {"type": "string"}},
                            "required": ["customer_id"]}}]
question = "Which orders of customer C002 are still shipped (in transit)?"
for name, tools, fns in [("A: dump all", TOOL_A, {"list_all_orders": list_all_orders}),
                         ("B: search", TOOL_B, {"search_orders": search_orders})]:
    before = len(tracker.calls)
    res = run_agent([{"role": "user", "content": question}], tools, fns, model=MODEL, effort="low", verbose=False)
    used = sum(c["input"] for c in tracker.calls[before:])
    print(f"{name:<12} input tokens={used:>6,}  steps={res.steps}  answer: {res.text[:90]!r}")

## 🆓 Free swap (optional)
The same tool loop in the OpenAI wire format (Gemini/Groq). Notice `tool_calls` and `role: "tool"`.

In [ ]:
import os
from course_utils import free_run_agent
provider = "gemini" if os.getenv("GEMINI_API_KEY") else "groq" if os.getenv("GROQ_API_KEY") else None
if provider:
    print(free_run_agent(provider, "What's the status of order O2001?", [GET_ORDER], {"get_order": get_order}))
else:
    print("No free-provider key set; skipping.")

## ✏️ Exercises
1. Remove `"is_error"` from the results in section 3. Does the model still recover from O9999?
2. Make `get_order`'s error message unhelpful (`"Error 404"`). Does the model behave worse?
3. Add a `check_refund_eligibility` tool that encodes the 30-day rule in code. Why is code better than a prompt rule here?

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker`. Make a habit of checking cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None